# Memory Management

Run this notebook cell-by-cell against the live cluster spawned from the topic page (uses this topic's 1-worker/8GB `cluster_defaults` -- the same "scaled up" allowance other spill/skew topics use). See `concept.md` for the what/why background.

Claims under test, each checked directly against real evidence rather than assumed:
1. `.cache()` + `.count()` on a ~3GB feature table shows fully cached on the Storage tab -- same materialization-confirmation pattern as `caching-persistence` (US-C5).
2. A competing memory-hungry shuffle (execution memory) evicts some of that cached storage -- measured as a real before/after drop in fraction-cached and per-executor storage `memoryUsed`, not described in prose.
3. Re-running the original cached query afterward shows a genuine **partial-recompute signal**: some partitions still instant (cached), others measurably recompute -- measured per-partition from real task durations, never hardcoded to the mockup's "3 of 8" example.
4. Connecting back to US-4.4's original spill/OOM-diagnosis criteria: a memory-constrained sort/aggregation produces nonzero spill metrics, and a deliberately oversized in-memory broadcast fails with a real `OutOfMemoryError` the notebook reads and reports rather than pre-diagnosing.

In [ ]:
import json
import time
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("memory-management").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)


def _get(path):
    url = f"http://localhost:4040/api/v1/applications/{app_id}/{path}"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def storage_snapshot():
    """Current `/storage/rdd` list -- same helper/disposition as
    `caching-persistence`'s `storage_snapshot()`: no fetcher for this
    endpoint exists in app/spark_api/app_client.py and the annotation
    manifest schema has no storage-shaped rule type, so this topic's
    fraction-cached/which-partitions-cached evidence is queried directly."""
    return _get("storage/rdd")


def executors_snapshot():
    """Current `/executors` list -- the same REST surface the new
    executor_metrics manifest mechanism reads server-side for the Reveal
    self-check (Decision A), queried directly here so the notebook itself
    can print a real, measured before/after `memoryUsed` comparison rather
    than deferring the whole story to a UI click."""
    return _get("executors")


def latest_stage_task_durations():
    """Per-partition (task `index`) wall-clock `duration` (ms) for the most
    recently submitted stage -- the real evidence for 'which partitions were
    still cached vs. recomputed', not assumed from the mockup's example."""
    stages = _get("stages")
    latest = max(stages, key=lambda s: s["stageId"])
    tasks = _get(f"stages/{latest['stageId']}/{latest.get('attemptId', 0)}/taskList?length=1000")
    return {t["index"]: t["duration"] for t in tasks if isinstance(t, dict) and "index" in t}


SCRATCH_DIR = "/workspace/scratch/memory-management"

## 1. Cache a ~3GB feature table across 8 partitions

Sized and partitioned (8 partitions) to match the mockup's "3 of 8 partitions evicted" walkthrough shape -- built the same vectorized way as `caching-persistence`'s wide-column table (column expressions, not `rdd.map(Row(...))`, which OOM'd executors outright when tried for real). High-cardinality double columns keep the columnar cache from compressing this away.

**Hypothesis:** before caching, does `/storage/rdd` show anything for this table? After `.cache()` + `.count()`, what fraction do you expect cached?

In [ ]:
FEATURE_ROWS = 24_000_000
NUM_DOUBLE_COLS = 16
FEATURE_PARTITIONS = 8

feature_df = spark.range(FEATURE_ROWS, numPartitions=FEATURE_PARTITIONS).select(
    F.col("id"),
    *[
        ((F.col("id") * 999983 + F.lit(j * 7919)) % 1_000_003).cast("double").alias(f"c{j}")
        for j in range(NUM_DOUBLE_COLS)
    ],
).cache()

before = storage_snapshot()
assert len(before) == 0, "expected nothing cached yet -- .cache() is lazy"

row_count = feature_df.count()
print(f"feature_df.count() = {row_count} rows across {FEATURE_PARTITIONS} partitions")

after = storage_snapshot()
assert len(after) == 1, f"expected exactly one cached RDD entry, got {len(after)}"
entry = after[0]
fraction_cached = entry["numCachedPartitions"] / entry["numPartitions"]
print(f"Fraction cached: {entry['numCachedPartitions']}/{entry['numPartitions']} = {fraction_cached:.0%}")
print(f"Memory used: {entry['memoryUsed'] / 1e9:.2f} GB")
assert fraction_cached == 1.0, "expected the feature table fully cached before any competing shuffle"

checkpoint(feature_df, topic="memory-management")
print("Checkpoint written -- click 'Reveal self-check' on the topic page now, before the competing shuffle below.")

## 2. Baseline: per-partition timing for a read against the fully-cached table

Records how fast each of the 8 partitions reads *while still fully cached*, so step 4 has something real to compare against instead of an assumed "fast" number.

In [ ]:
feature_df.agg(F.sum("c0")).collect()
baseline_durations = latest_stage_task_durations()
print(f"Baseline per-partition durations (ms), all fully cached: {sorted(baseline_durations.values())}")

## 3. Run a competing memory-hungry shuffle for execution memory

A large sort over a *separate*, uncached, multi-GB dataset on this same 1-worker/8GB cluster -- genuinely competes with `feature_df`'s cached storage blocks for the same unified memory pool.

**Hypothesis:** will this shuffle need to evict any of `feature_df`'s cached partitions? What do you expect to happen to `memoryUsed` on the executor, and to the Storage tab's fraction cached?

In [ ]:
SHUFFLE_ROWS = 40_000_000
shuffle_df = spark.range(SHUFFLE_ROWS, numPartitions=32).select(
    F.col("id"),
    ((F.col("id") * 15485863) % 10_000_019).cast("double").alias("sort_key"),
)
# Few shuffle partitions -> each execution-memory buffer this sort needs is
# large, forcing genuine contention against feature_df's cached storage
# instead of both fitting comfortably side by side.
spark.conf.set("spark.sql.shuffle.partitions", "4")

executors_before = executors_snapshot()
storage_before = storage_snapshot()

shuffle_df.orderBy("sort_key").foreach(lambda row: None)

executors_after = executors_snapshot()
storage_after = storage_snapshot()


def total_storage_memory_used(snapshot):
    return sum(e.get("memoryUsed", 0) or 0 for e in snapshot if e.get("id") != "driver")


mem_before = total_storage_memory_used(executors_before)
mem_after = total_storage_memory_used(executors_after)
print(f"Executor storage memoryUsed before competing shuffle: {mem_before / 1e6:.1f} MB")
print(f"Executor storage memoryUsed after competing shuffle:  {mem_after / 1e6:.1f} MB")

fraction_before = storage_before[0]["numCachedPartitions"] / storage_before[0]["numPartitions"]
fraction_after = storage_after[0]["numCachedPartitions"] / storage_after[0]["numPartitions"]
print(f"Fraction cached before: {fraction_before:.0%}  after: {fraction_after:.0%}")

evicted_measured = fraction_after < fraction_before or mem_after < mem_before
print(f"Measured eviction from the competing shuffle: {evicted_measured}")
if not evicted_measured:
    print(
        "No eviction observed -- if this cluster's total memory comfortably fits both the "
        "feature table and the shuffle's execution-memory need, re-run with a larger "
        "SHUFFLE_ROWS or fewer shuffle.partitions to force real contention."
    )

## 4. Re-run the cached query -- measure the partial-recompute signal

**Hypothesis:** will every partition read instantly (nothing evicted), will every partition recompute (everything evicted), or a genuine mix? Compare each partition's duration below against step 2's baseline -- the mockup's own target shape is "3 of 8 partitions evicted", but this cell measures the real fraction for this run rather than assuming that number.

In [ ]:
feature_df.agg(F.sum("c0")).collect()
rerun_durations = latest_stage_task_durations()

# A partition counts as "recomputed" if its duration is well above the
# slowest fully-cached baseline observation -- a generous multiplier (not a
# hardcoded absolute threshold) so this holds regardless of the cluster's
# absolute speed on a given run.
baseline_ceiling_ms = max(baseline_durations.values()) * 3
still_cached = [idx for idx, d in rerun_durations.items() if d <= baseline_ceiling_ms]
recomputed = [idx for idx, d in rerun_durations.items() if d > baseline_ceiling_ms]

print(f"Baseline ceiling for 'still cached': {baseline_ceiling_ms:.0f}ms")
print(f"Rerun per-partition durations (ms): {rerun_durations}")
print(f"Partitions still cached (fast): {sorted(still_cached)}")
print(f"Partitions recomputed (slow): {sorted(recomputed)}")
print(f"Measured: {len(recomputed)} of {len(rerun_durations)} partitions evicted and recomputed.")

checkpoint(feature_df, topic="memory-management")
print(
    "Checkpoint re-written -- click 'Reveal self-check' again: the executor_metrics evidence "
    "below the plan/stage panels now reflects the post-shuffle memoryUsed, not the pre-shuffle "
    "snapshot from step 1."
)

## 5. Spill under memory-constrained aggregation (connects back to US-4.4)

US-4.4's original spill/OOM-diagnosis criteria are extended, not relaxed, by this topic. Same technique as step 3 (very few shuffle partitions forces a large per-partition sort buffer), pushed further so it genuinely spills to disk rather than merely evicting cache.

**Hypothesis:** will `memoryBytesSpilled`/`diskBytesSpilled` be nonzero for this stage? Locate them in the Spark UI's Stages tab (or the self-check's stage-metrics panel below, which spotlights both) before trusting this cell's own printout.

In [ ]:
SPILL_ROWS = 60_000_000
spill_df = spark.range(SPILL_ROWS, numPartitions=48).select(
    F.col("id"),
    ((F.col("id") * 2654435761) % 100_000_007).cast("double").alias("sort_key"),
)
spark.conf.set("spark.sql.shuffle.partitions", "2")  # 2 huge partitions -- forces real spill, not just eviction

spill_df.orderBy("sort_key").foreach(lambda row: None)

stages = _get("stages")
latest_stage = max(stages, key=lambda s: s["stageId"])
memory_spilled = latest_stage.get("memoryBytesSpilled", 0) or 0
disk_spilled = latest_stage.get("diskBytesSpilled", 0) or 0
print(f"Stage {latest_stage['stageId']}: memoryBytesSpilled={memory_spilled}, diskBytesSpilled={disk_spilled}")
if memory_spilled == 0 and disk_spilled == 0:
    print(
        "No spill observed on this run -- increase SPILL_ROWS or lower shuffle.partitions "
        "further to force it on a cluster with more available memory than this notebook assumed."
    )

spark.conf.set("spark.sql.shuffle.partitions", "200")  # restore default

## 6. Diagnosing OOM: forcing a broadcast past a safe memory threshold (connects back to US-4.4)

`.hint("broadcast")` forces Spark to attempt broadcasting `feature_df` (multiple GB) to every executor, bypassing `spark.sql.autoBroadcastJoinThreshold`'s own safety check entirely -- a deliberately oversized in-memory operation on a deliberately memory-constrained (single, 8GB) executor. **This is expected to fail** with a real `OutOfMemoryError`; the goal is reading and diagnosing that failure yourself, not avoiding it. If it takes the sole executor down with it, respawn the cluster from the topic page before continuing to step 7 -- this notebook does not pre-diagnose which memory region was exhausted for you.

In [ ]:
tiny_df = spark.range(10).select(F.col("id").alias("key"))
oversized_broadcast_df = feature_df.hint("broadcast")

try:
    tiny_df.join(oversized_broadcast_df, tiny_df.key == oversized_broadcast_df.id).count()
    print("No OOM observed -- this cluster had enough memory to build the broadcast; increase "
          "FEATURE_ROWS/NUM_DOUBLE_COLS in step 1 to force it on a larger cluster.")
except Exception as exc:
    message = str(exc)
    print("Broadcast join failed, as expected. Raw driver-side error (read this, don't skip it):\n")
    print(message[:2000])
    if "OutOfMemoryError" in message or "Not enough memory" in message:
        print(" Diagnosis: execution memory (broadcast build/collection on driver or executor) was exhausted -- read which side the raw error above names.")

## 7. Clean up

Best-effort -- if step 6 took the executor down, this cell may itself fail; that is expected, respawn the cluster instead of treating a cleanup failure here as a bug.

In [ ]:
try:
    feature_df.unpersist(blocking=True)
    spark.conf.set("spark.sql.shuffle.partitions", "200")
except Exception as exc:
    print(f"Cleanup skipped (cluster likely needs a respawn after step 6's OOM): {exc}")